In [1]:
import os
os.chdir('/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/')

# general
import glob
import datetime as dt
from pathlib import Path

# data 
import xarray as xr 
import numpy as np
import pandas as pd

# plotting
import matplotlib.pyplot as plt
import plotly.express as px 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Configure Plotly for Jupyter notebooks
pio.renderers.default = "notebook"
# Alternative renderers you can try if "notebook" doesn't work:
# pio.renderers.default = "plotly_mimetype+notebook"
# pio.renderers.default = "jupyter_lab"

# helper tools
from metpy import calc, units
import scipy.stats as stats
from sklearn.linear_model import LinearRegression

In [2]:
DATA_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis'
files = glob.glob(os.path.join(DATA_DIR, '*.nc'))

# Define your root and sites
DATA_DIR = Path(DATA_DIR)
sites = ["gothic", "kettle_ponds"]

# Define all possible subfolder patterns
subfolders = {
    ("gridded", "with_normalized_met"): "gridded_events_with_normalized_met",
    ("gridded", "with_raw_met"): "gridded_events_with_raw_met",
    ("gridded", None): "gridded_events",
    (None, "with_normalized_met"): "events_with_normalized_met",
    (None, "with_raw_met"): "events_with_raw_met",
    (None, None): "events",
}

# Make directories up front
for site in sites:
    for folder in set(subfolders.values()):
        (DATA_DIR / site / folder).mkdir(parents=True, exist_ok=True)

# Move files
for file in files:
    file_path = Path(file)
    for site in sites:
        if site in file_path.name:
            # Identify flags in filename
            is_gridded = "gridded" if "gridded" in file_path.name else None
            met_flag = None
            if "with_normalized_met" in file_path.name:
                met_flag = "with_normalized_met"
            elif "with_raw_met" in file_path.name:
                met_flag = "with_raw_met"

            # Determine destination folder
            dest_folder = subfolders.get((is_gridded, met_flag))
            if dest_folder is None:
                raise ValueError(f"No match for file {file_path.name}")

            dest_path = DATA_DIR / site / dest_folder / file_path.name
            # print(f"Moving {file_path} to {dest_path}")
            os.rename(file_path, dest_path)
            break  # Done with this site
# update files so it does not try to move again
files = glob.glob(os.path.join(DATA_DIR, '*.nc'))
print(f"Total files remaining in {DATA_DIR}: {len(files)}")

Total files remaining in /storage/dlhogan/precipitation-rodeo/data/for_analysis: 0


In [ ]:
def get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED):
    if WITH_MET and RAW_OR_NORMALIZED == "raw":
        WITH_MET = "_with_raw_met"
    elif WITH_MET and RAW_OR_NORMALIZED == "normalized":
        WITH_MET = "_with_normalized_met"
    else:
        WITH_MET = ""

    if PRODUCT == "gridded":
        PRODUCT_NAME = "_gridded"
        FOLDER_NAME = "gridded_events"
    elif PRODUCT == "events":
        PRODUCT_NAME = ""
        FOLDER_NAME = "events"
    else:
        PRODUCT_NAME = ""
    if SRC in ['asfs', 'sos']:
        SITE = 'kettle_ponds'
    elif SRC in ['bb', 'sail']:
        SITE = 'gothic'
    else:
        site = input("Enter site name (gothic or kettle_ponds): ")
    return DATA_DIR / SITE / f"{FOLDER_NAME}{WITH_MET}" / f"{SITE}{PRODUCT_NAME}_precipitation_event_comparisons_{SRC}{WITH_MET}.nc"

In [4]:
SRC = "sail" # one of [bb, sail, asfs, sos, '']
PRODUCT = "events" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "raw"  # or raw
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/gothic/events_with_raw_met/gothic_precipitation_event_comparisons_sail_with_raw_met.nc


Let's plot the recurrence of these events by month, what is their distribution? How many events occur per month?

In [17]:
ds = ds.sel(event_id=slice(1,11))
monthly_counts_list = []
for ins in ds.sel(benchmark="billy_barr_precip")['test_instrument'].values:
    if ins == 'billy_barr_precip':
        continue
    tmp_ds = ds.sel(benchmark="billy_barr_precip", test_instrument=ins)
    
    tmp_ds = tmp_ds.assign_coords(
        month=("event_id", tmp_ds["start_time"].dt.month.data)
    )
    grouped = tmp_ds['start_time'].groupby(["test_instrument", "month"])
    monthly_counts = grouped.count()
    monthly_counts_list.append(monthly_counts)
monthly_counts_ds = xr.concat(monthly_counts_list, dim="test_instrument")

In [22]:
# Plot the counts for each instrument by month as a line plot
fig = go.Figure()
# create continuous viridis color scale
color_scale = px.colors.qualitative.Dark24
# create the length to match number of instruments

for ins in monthly_counts_ds['test_instrument'].values:
    fig.add_trace(go.Scatter(
        x=monthly_counts_ds.sel(test_instrument=ins)['month'],
        y=monthly_counts_ds.sel(test_instrument=ins).values,
        mode='lines+markers',
        name=ins,
        line=dict(color=color_scale[hash(ins) % len(color_scale)])
    ))
fig.update_layout(
    title='Monthly Event Counts by Instrument (Gothic Site)',
    xaxis_title='Month',
    yaxis_title='Number of Events',
    xaxis=dict(tickmode='array', tickvals=list(range(1, 13)), ticktext=[
        'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
    ]),
    height=600,
    width=1000,
)
# change y axis range from 0, to max + 3
y_max = monthly_counts_ds.max().values + 1
fig.update_yaxes(range=[0, y_max])
# make the background dark
# fig.update_layout(plot_bgcolor='rgba(0, 0, 0, 0)', paper_bgcolor='rgba(0, 0, 0, 0)')
fig.show()

In [9]:
SRC = "asfs" # one of [bb, sail, asfs, sos, '']
PRODUCT = "events" # or events
WITH_MET = True  # or True
RAW_OR_NORMALIZED = "normalized"  # or raw
file_dest = get_file_destination(DATA_DIR, SRC, PRODUCT, WITH_MET, RAW_OR_NORMALIZED)
print(f"Loading data from: {file_dest}")
ds = xr.open_dataset(file_dest)

Loading data from: /storage/dlhogan/precipitation-rodeo/data/for_analysis/kettle_ponds/events_with_normalized_met/kettle_ponds_precipitation_event_comparisons_asfs_with_normalized_met.nc


In [10]:
monthly_counts_list = []
for ins in ds.sel(benchmark="billy_barr_precip")['test_instrument'].values:
    if ins == 'billy_barr_precip':
        continue
    tmp_ds = ds.sel(benchmark="billy_barr_precip", test_instrument=ins)
    
    tmp_ds = tmp_ds.assign_coords(
        month=("event_id", tmp_ds["start_time"].dt.month.data)
    )
    grouped = tmp_ds['start_time'].groupby(["test_instrument", "month"])
    monthly_counts = grouped.count()
    monthly_counts_list.append(monthly_counts)
monthly_counts_ds = xr.concat(monthly_counts_list, dim="test_instrument")

In [11]:
# Plot the counts for each instrument by month as a line plot
fig = go.Figure()
for ins in monthly_counts_ds['test_instrument'].values:
    fig.add_trace(go.Scatter(
        x=monthly_counts_ds.sel(test_instrument=ins)['month'],
        y=monthly_counts_ds.sel(test_instrument=ins).values,
        mode='lines+markers',
        name=ins
    ))
fig.update_layout(
    title='Monthly Event Counts by Instrument (Kettle Ponds Site)',
    xaxis_title='Month',
    yaxis_title='Number of Events',
    xaxis=dict(tickmode='array', tickvals=list(range(1, 13)), ticktext=[
        'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
    ])
)
fig.show()